In [ ]:
# --- Xray Cloud: Release Test Report using Test Execution Key + Daily Snapshot & Csv ---

import requests, getpass, json, collections, datetime, os, csv

TEST_EXEC_KEY = "PROJ-2001"   # <-- put your Test Execution key here

XRAY_AUTH_URL    = "https://xray.cloud.getxray.app/api/v2/authenticate"
XRAY_GRAPHQL_URL = "https://xray.cloud.getxray.app/api/v2/graphql"

# TEMP: bypass SSL while you're unblocking (replace with a CA bundle later)
VERIFY_SSL = False
if not VERIFY_SSL:
    from urllib3.exceptions import InsecureRequestWarning
    import urllib3; urllib3.disable_warnings(InsecureRequestWarning)

# ---- prompt for Xray API keys (NOT Atlassian PAT) ----
CID  = getpass.getpass("XRAY_CLIENT_ID (Xray → API Keys): ")
CSEC = getpass.getpass("XRAY_CLIENT_SECRET (Xray → API Keys): ")

def auth():
    r = requests.post(XRAY_AUTH_URL, json={"client_id": CID, "client_secret": CSEC},
                      timeout=30, verify=VERIFY_SSL)
    if r.status_code != 200:
        raise SystemExit(f"Auth failed: {r.status_code} {r.text[:300]}")
    return r.text.strip('"')

TOKEN   = auth()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

def gql(query, variables=None):
    r = requests.post(XRAY_GRAPHQL_URL, json={"query": query, "variables": variables or {}},
                      headers=HEADERS, timeout=60, verify=VERIFY_SSL)
    if r.status_code != 200:
        raise SystemExit(f"GraphQL HTTP {r.status_code}: {r.text[:300]}")
    data = r.json()
    if "errors" in data:
        raise SystemExit(f"GraphQL errors: {data['errors']}")
    return data["data"]

# 1) Resolve Test Execution KEY -> issueId 
Q_EXEC = """
query($jql:String!, $limit:Int!, $start:Int!) {
  getTestExecutions(jql:$jql, limit:$limit, start:$start) {
    results { issueId jira(fields:["key","summary"]) }
  }
}
"""
exec_search = gql(Q_EXEC, {"jql": f'key = "{TEST_EXEC_KEY}"', "limit": 1, "start": 0})
if not exec_search["getTestExecutions"]["results"]:
    raise SystemExit(f"No Test Execution found for key {TEST_EXEC_KEY}")
EXEC_ID   = exec_search["getTestExecutions"]["results"][0]["issueId"]
EXEC_NAME = f'{exec_search["getTestExecutions"]["results"][0]["jira"]["key"]} - {exec_search["getTestExecutions"]["results"][0]["jira"]["summary"]}'

# 2) Page Test Runs for execution and count statuses
Q_RUNS = """
query($execIds:[String], $limit:Int!, $start:Int!) {
  getTestRuns(testExecIssueIds:$execIds, limit:$limit, start:$start) {
    total start limit
    results { status { name } }
  }
}
"""

counts = collections.Counter()
total_runs = 0
start = 0
limit = 100
while True:
    data = gql(Q_RUNS, {"execIds": [EXEC_ID], "limit": limit, "start": start})
    runs = data["getTestRuns"]
    for tr in runs["results"]:
        sname = (tr.get("status") or {}).get("name") or "UNKNOWN"
        counts[sname] += 1
        total_runs += 1
    start += runs["limit"]
    if start >= runs["total"]:
        break

# 3) Normalize and roll up to  buckets
norm = {}
for k, v in counts.items():
    nk = (k or "UNKNOWN").strip().upper()
    norm[nk] = norm.get(nk, 0) + v

PASSED    = sum(norm.get(k, 0) for k in ("PASSED", "PASS", "OK"))
FAILED    = sum(norm.get(k, 0) for k in ("FAILED", "FAIL"))
DESCOPED  = sum(norm.get(k, 0) for k in ("DESCOPED", "NOT EXECUTED (DESCOPED)", "SKIPPED (DESCOPED)"))
TODO      = sum(norm.get(k, 0) for k in ("TODO", "TO DO", "NOT_RUN", "UNTESTED"))

executed  = PASSED + FAILED + sum(norm.get(k, 0) for k in ("BLOCKED","EXECUTING","ABORTED"))  # include any other exec statuses you use
not_exec  = TODO + DESCOPED
pass_pct  = round((PASSED / executed * 100.0), 2) if executed else 0.0
completion= round((executed / total_runs * 100.0), 2) if executed else 0.0

summary = {
    "Test Execution": EXEC_NAME,
    "Totals (Test Runs)": {
        "Total": total_runs,
        "Passed": PASSED,
        "Failed": FAILED,
        "Descoped": DESCOPED,
        "To Do": TODO,
        "Executed": executed,
        "Not Executed": not_exec,
        "Pass % (of executed)": round(pass_pct, 2),
         "Completion %": completion,
    },
    "Other Statuses": {k: v for k, v in norm.items()
                       if k not in {"PASSED","PASS","OK","FAILED","FAIL","DESCOPED",
                                    "NOT EXECUTED (DESCOPED)","SKIPPED (DESCOPED)",
                                    "TODO","TO DO","NOT_RUN","UNTESTED","BLOCKED","EXECUTING","ABORTED"}}
}

ts = datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%d %H:%M UTC")
md = [
    f"### Test Execution Rollup — {EXEC_NAME}",
    f"_Generated: {ts}_",
    "",
    "| Metric | Count |",
    "|---|---|",
    f"| Total | {summary['Totals (Test Runs)']['Total']} |",
    f"| Passed | {summary['Totals (Test Runs)']['Passed']} |",
    f"| Failed | {summary['Totals (Test Runs)']['Failed']} |",
    f"| Descoped | {summary['Totals (Test Runs)']['Descoped']} |",
    f"| To Do | {summary['Totals (Test Runs)']['To Do']} |",
    f"| Executed | {summary['Totals (Test Runs)']['Executed']} |",
    f"| Not Executed | {summary['Totals (Test Runs)']['Not Executed']} |",
    f"| Pass % (of executed) | {summary['Totals (Test Runs)']['Pass % (of executed)']} |",
    f"| Completion % | {summary['Totals (Test Runs)']['Completion %']} |",
]
print("\n".join(md))

# ----------------------------
# Append a daily snapshot
# ----------------------------
def save_daily_snapshot(TEST_EXEC_KEY, totals):
    
    today = datetime.datetime.now().strftime("%Y-%m-%d")
    fname = f"Xray_Execution Test Report_{TEST_EXEC_KEY}.csv"

    row = {
        "date": today,
        "total": totals["Total"],
        "passed": totals["Passed"],
        "failed": totals["Failed"],
        "descoped": totals["Descoped"],
        "todo": totals["To Do"],  
        "Pass %": totals["Pass % (of executed)"],
        "Completion %": totals["Completion %"],
    }

    file_exists = os.path.exists(fname)
    with open(fname, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            w.writeheader()
        w.writerow(row)
    return fname

csv_path = save_daily_snapshot(TEST_EXEC_KEY, summary["Totals (Test Runs)"])
print(f"\n[+] Snapshot appended to {csv_path}")